# NB03 - Structures de données complexes

Les types de données complexes en Spark (et en Python) sont des structures qui permettent de regrouper plusieurs valeurs ou objets dans une seule entité. Contrairement aux types simples (entiers, chaînes de caractères, booléens), ils facilitent la gestion de données structurées ou hiérarchiques. Les principaux types complexes sont :

- Array (liste) : une collection ordonnée d’éléments du même type, accessible par index. Utile pour stocker des listes de valeurs, comme plusieurs tags ou notes pour un utilisateur.

- Map (dictionnaire) : une collection de paires clé/valeur, permettant d’associer une valeur à une clé unique. Pratique pour représenter des attributs dynamiques ou des configurations.

- Struct (structure) : un ensemble de champs nommés, chacun pouvant avoir un type différent. Cela permet de regrouper plusieurs propriétés liées dans un même objet, comme une adresse composée de rue, ville et code postal.


Ces types sont essentiels pour manipuler des données semi-structurées (JSON, Parquet) et pour effectuer des transformations avancées dans Spark DataFrame. Ils facilite

## A. Chargement des données et préparation du notebook

%md
%md
Dans ce notebook il y a un paramètre `full_path` en haut du notebook

![](./Resources/NB03/img_parameters.png)

La cellule dessous permet de créer ce paramètre si il n'existe pas, il y aura une valeur par défaut vers l'emplacement du fichier MOCK_DATA.csv : `/Workspace/Users/{username}/databricks-training/Spark Developer/Développement d'applications Spark/users_dataset.csv`

Comme Spark ne peut pas lire de fichier directement dans le système de fichier du Workspace comme on l'as fait avec Pands, il faut : 
- Créer un volume (dans un Catalog/schema)
- Placer le fichier dans le volume

Les deux prochaines cellules vont paramètrer les différentes variables nécéssaires pour le `Setup`. 

Par défaut un volume sera créer dans le catalog : **workspace.default** (spark_training) mais on peut modifier les widgets (directement dans le code ou si la cellules à déjà été exécuté en haut du notebook).

In [0]:
# Passe la variable 'full_path' avec le chemin vers le fichier CSV MOCK_DATA.csv au notebook NB02/Setup
import os

full_path = os.getcwd() + "/Resources/NB03/users_dataset.csv"

dbutils.widgets.text("full_path", full_path)
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("volume", "spark_training")
dbutils.widgets.text("filename", "users_dataset.csv")

In [0]:
%run "./Resources/NB03/Setup"

La table users_dataset devrait maintenant être disponible

In [0]:
%sql
select * from workspace.default.users_dataset;

## B. Convertion de JSON vers StructType

La conversion d’un JSON vers un `StructType` permet à Spark de comprendre la structure des données et de les manipuler efficacement. Un `StructType` définit explicitement les champs, leurs types et leur organisation, ce qui facilite :

- La validation des données lors du chargement
- L’accès et la transformation des champs individuels
- L’optimisation des requêtes et des opérations Spark

En résumé, cela permet de travailler avec des données semi-structurées (comme le JSON) de façon structurée et performante dans Spark.

### Etapes de la conversion
1. **Définir le schéma StructType**  ou **InferSchema**
   avec la méthode (`schema_of_json`)
   On crée un objet `StructType` en spécifiant les champs et leurs types (ex : `StructField("nom", StringType(), True)`).

2. **Charger le JSON**  
   On lit le fichier ou la colonne JSON avec Spark (ex : `spark.read.json(path)` ou `from_json(col, schema)`).

3. **Appliquer le schéma**  
   On utilise le schéma StructType pour parser et structurer les données JSON (ex : `df.withColumn("data_struct", from_json("json_col", schema))`).

4. **Accéder aux champs structurés**  
   Les champs du JSON sont accessibles comme colonnes du DataFrame, ce qui permet des transformations et des requêtes efficaces.

5. **Valider et transformer les données**  
   Le schéma permet de valider les types et de manipuler les données de façon structurée (filtrage, sélection, agrégation).

   ### Avantages des StructTypes

- Validation automatique des types et de la structure des données
- Accès facile et performant aux champs individuels
- Optimisation des requêtes Spark grâce à une structure explicite
- Manipulation efficace des données semi-structurées (JSON, Parquet)
- Facilite les transformations, filtrages et agrégations complexes

In [0]:
raw_users_data_df = spark.read.table(f"{catalog}.{schema}.users_dataset")

In [0]:
# inspection du schéma
raw_users_data_df.printSchema()

In [0]:
display(raw_users_data_df)

1. la colonne **interests** est de type `Array`
2. la colonne **recent_purchases** est du JSON

In [0]:
from pyspark.sql.types import ArrayType, StringType

interests_schema = ArrayType(StringType())

In [0]:
recent_purchases_json = raw_users_data_df.select("recent_purchases").limit(1).collect()[0][0]
print(recent_purchases_json)

La fonction `schema_of_json` en Spark permet d'inférer automatiquement le schéma d'un JSON sous forme de chaîne de caractères. Elle retourne un objet `StructType` qui décrit la structure du JSON (noms des champs, types, etc.). Cela est utile pour parser des colonnes JSON avec la fonction `from_json` sans avoir à définir manuellement le schéma.

**Exemple d'utilisation :**
```python
from pyspark.sql.functions import schema_of_json

# Récupérer un exemple de JSON depuis une colonne
json_sample = raw_users_data_df.select("recent_purchases").limit(1).collect()[0][0]

# Inférer le schéma
json_schema = schema_of_json(json_sample)
print(json_schema)
```


Cette méthode facilite la conversion de données JSON en colonnes structurées dans un DataFrame Spark.

In [0]:
from pyspark.sql.functions import schema_of_json, lit

recent_purchases_schema = schema_of_json(lit(recent_purchases_json))

In [0]:
recent_purchases_schema

In [0]:
from pyspark.sql.functions import col, from_json

parsed_users_df = raw_users_data_df.select(
    col("user_id"),
    col("name"),
    col("active"),
    from_json(col("interests"), interests_schema).alias("interests"),
    from_json(col("recent_purchases"), recent_purchases_schema).alias("recent_purchases")
)

parsed_users_df.printSchema()

In [0]:
display(parsed_users_df)

## C. Arrays

In [0]:
from pyspark.sql.functions import array_size

display(
    parsed_users_df.select(
        "user_id",
        array_size("interests").alias("nb_interests"),
        array_size("recent_purchases").alias("nb_recent_purchases")
    )
)

### 1. `Explode`

La méthode `explode` en Spark permet de transformer une colonne de type `Array` ou `Map` en plusieurs lignes, une par élément de l'array ou du map. Cela facilite l'analyse des données imbriquées en les "aplatissant".

**Exemple :**
Si une colonne `interests` contient `["music", "sports"]`, l'utilisation de `explode("interests")` produira deux lignes : une avec "music", une avec "sports".

**Syntaxe :**
```python
from pyspark.sql.functions import explode

df.select("user_id", explode("interests").alias("interest"))
```

In [0]:
# prenons par exemple l'utilisateur avec l' id U001
user_U001_interests_df = parsed_users_df.select("user_id", "interests").filter(parsed_users_df.user_id == "U001")
display(user_U001_interests_df)

In [0]:
# on peut aussi utiliser la fonction explode pour décomposer les données dans un dataframe
# on a donc 1 ligne pour chaque élément de l'array
from pyspark.sql.functions import explode

display(
    user_U001_interests_df.select(
        "user_id",
        explode("interests").alias("interest")
    )
)

### 2. `collect_set` et `collect_list`

Les fonctions `collect_set` et `collect_list` sont utilisées pour agréger des valeurs dans Spark lors de groupBy, en particulier pour regrouper plusieurs lignes en une seule liste ou ensemble.

- **`collect_list`** : Agrège les valeurs d'une colonne en une liste (avec doublons possibles).
  - Exemple : Si plusieurs lignes ont la même valeur de `user_id`, `collect_list("interest")` retourne une liste de tous les intérêts, y compris les répétitions.

- **`collect_set`** : Agrège les valeurs d'une colonne en un ensemble (sans doublons).
  - Exemple : `collect_set("interest")` retourne une liste unique des intérêts pour chaque `user_id`, en supprimant les doublons.

**Syntaxe :**
```python
from pyspark.sql.functions import collect_list, collect_set

df.groupBy("user_id").agg(
    collect_list("interest").alias("all_interests"),
    collect_set("interest").alias("unique_interests")
)
```

In [0]:
# on va créer un dataset avec la colonne interests explosée
exploded_users_df = parsed_users_df.select(
    "user_id",
    explode("interests").alias("interest")
)

display(exploded_users_df)

In [0]:
# on utilise la méthide `collect_list` pour récupérer tout les interests d'un utilisateur
from pyspark.sql.functions import collect_list

display(
    exploded_users_df.groupBy("user_id").agg(
        collect_list("interest").alias("interests")
    )
)

## D. Référence à des champs d'un objet Struct

In [0]:
exploded_purchases_df = parsed_users_df.select(
    "user_id",
    explode("recent_purchases").alias("purchase")
)

display(exploded_purchases_df)

pour acceder au champ d'un Struct on utilise :
- la notation "."
- la méthode getField(str)

In [0]:
recent_purchases_df = exploded_purchases_df.select(
    "user_id",
    col("purchase.name").alias("purchase_name"),
    col("purchase.product_id").alias("purchase_pid"),
    col("purchase.price").alias("purchase_price")
)

display(recent_purchases_df)

In [0]:
recent_purchases_df = exploded_purchases_df.select(
    "user_id",
    col("purchase").getField("name").alias("purchase_name"),
    col("purchase").getField("product_id").alias("purchase_pid"),
    col("purchase").getField("price").alias("purchase_price")
)

display(recent_purchases_df)

## E. Utilisation du Pivot

La méthode `pivot` en Spark permet de transformer les valeurs d'une colonne en plusieurs colonnes, facilitant l'analyse croisée (tableau croisé dynamique). On l'utilise généralement après un `groupBy` pour agréger des données selon différentes catégories.

**Exemple :**
```python
df.groupBy("user_id").pivot("interest").count()
```

Ici, chaque valeur unique de `interest` devient une colonne, avec le nombre d'occurrences pour chaque `user_id`.

In [0]:
from pyspark.sql.functions import count

pivot_df = (recent_purchases_df
    .groupBy("user_id")
    .pivot("purchase_name")
    .agg(count("purchase_pid").alias("quantity_purchased"))
)

display(pivot_df)

In [0]:
# on remplace les null par 0 pour plus de lisibilité
pivot_df = (recent_purchases_df
    .groupBy("user_id")
    .pivot("purchase_name")
    .agg(count("purchase_pid").alias("quantity_purchased"))
    .fillna(0)
)

display(pivot_df)